# Phoneme-based PD Classification
Classification of Parkinson's Disease vs healthy controls using phoneme duration statistics.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Define Data Root relative to the notebook
DATA_ROOT = '../datalocal/PC-GITA_v260210_24kHz/'

# Add root directory to sys.path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from phoneme_evaluation.data_loader import load_all_data, get_speaker_features
from phoneme_evaluation.constants import TASKS

In [2]:
# Load all data
print("Loading data...")
df_all = load_all_data()
feature_df = get_speaker_features(df_all)
print(f"Loaded {len(feature_df)} speakers with {feature_df.shape[1]-2} features.")

Loading data...
Loading metadata from C:\Users\smidl\smidl_local\work\FAU\dysarthric-TTS\datalocal\PC-GITA_v260210_24kHz\_metadata\PCGITAtoPD_mapping.csv...
Processing directories: ['ddk', 'monologue_split', 'readtext_split', 'sentences_cleaned', 'words_merged']
 - Found 150 files in ddk
 - Found 793 files in monologue_split
 - Found 550 files in readtext_split
 - Found 890 files in sentences_cleaned
 - Found 430 files in words_merged
Filtering technical tokens: ['<p:>', '<usb>', 'sil', 'sp', 'SIL', 'SP']
Filtering outliers with Z-threshold=3.0...
Total samples loaded: 98647
Loaded 100 speakers with 66 features.


## Feature Vector Construction

The classification is performed at the **speaker level**. For each speaker, we construct a single feature vector from their aggregated phoneme alignment data.

### How it is constructed:
1. **Aggregation:** We take all recordings for a specific speaker within the selected task.
2. **Feature Extraction:** For every unique phoneme, we calculate the **Mean Duration** and **Variance of Duration** across all occurrences.
3. **Vectorization:** These values are pivoted into a wide-format vector: [mean_a, var_a, mean_b, var_b, ...] .
4. **Missing Values:** If a speaker didn't produce a specific phoneme, the value is filled with **0**.

### Classification Level:
The script classifies **each speaker once** (one vector per person). It does NOT classify individual recordings separately. This ensures that the model learns stable acoustic traits rather than transient noise.

In [3]:
# Example of a single speaker's feature vector
example_speaker = feature_df.iloc[0]
print(f'Speaker ID: {feature_df.index[0]}')
print('Status:', example_speaker['status'])
print('Vector Dimension:', len(example_speaker.drop(['status', 'sex'])))
print('\\nFirst 10 features (phoneme means/vars):')
print(example_speaker.drop(['status', 'sex']).head(10))

Speaker ID: 001PD
Status: PD
Vector Dimension: 66
\nFirst 10 features (phoneme means/vars):
mean_B    0.041863
mean_D    0.065625
mean_F         0.0
mean_G    0.049958
mean_J    0.075158
mean_L    0.041458
mean_N    0.099958
mean_S         0.0
mean_T    0.059958
mean_a    0.101212
Name: 001PD, dtype: object


In [4]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

def run_classification_experiment(df, title):
    X = df.drop(columns=['status', 'sex'])
    y = df['status'].apply(lambda x: 1 if x == 'PD' else 0)
    groups = df.index
    
    # Print class distribution and dimension
    counts = df['status'].value_counts().to_dict()
    print(f'--- {title} ---')
    print(f'Class Distribution (Speakers): {counts}')
    print(f'Feature Dimension: {X.shape[1]}')
    
    n_splits = max(2, min(10, min(counts.values())))
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'MLP': MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500, random_state=42),
        'HGBT': HistGradientBoostingClassifier(random_state=42)
    }
    
    results = []
    cv = StratifiedGroupKFold(n_splits=n_splits)
    
    for name, model in models.items():
        pipeline = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        y_true, y_pred, y_prob = [], [], []
        try:
            for train_idx, test_idx in cv.split(X, y, groups=groups):
                pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
                y_true.extend(y.iloc[test_idx])
                y_pred.extend(pipeline.predict(X.iloc[test_idx]))
                if hasattr(model, 'predict_proba'):
                    probs = pipeline.predict_proba(X.iloc[test_idx])[:, 1]
                else:
                    probs = pipeline.predict(X.iloc[test_idx])
                y_prob.extend(probs)
            
            results.append({
                'Model': name,
                'Accuracy': accuracy_score(y_true, y_pred),
                'F1': f1_score(y_true, y_pred),
                'AUC': roc_auc_score(y_true, y_prob)
            })
        except Exception as e: print(f'Error {name}: {e}')
    
    res_df = pd.DataFrame(results)
    display(res_df.style.format(precision=4))
    return res_df

In [5]:
# Global Experiment
run_classification_experiment(feature_df, "All Tasks Combined")

# Per-Task Experiments
task_results = {}
for task in TASKS:
    df_task = df_all[df_all['task'] == task]
    if df_task.empty: continue
    
    feat_task = get_speaker_features(df_task)
    if len(feat_task) < 20: continue # Skip if too few speakers
    
    task_results[task] = run_classification_experiment(feat_task, f"Task: {task}")

--- All Tasks Combined ---
Class Distribution (Speakers): {'PD': 50, 'HC': 50}
Feature Dimension: 66


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.8200,0.8200,0.8920
1,MLP,0.7900,0.7835,0.8848
2,HGBT,0.7600,0.7600,0.8328


--- Task: ddk ---
Class Distribution (Speakers): {'PD': 50}
Feature Dimension: 26
Error Logistic Regression: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)


a:\work\FAU\dysarthric-TTS\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
a:\work\FAU\dysarthric-TTS\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Model,Accuracy,F1,AUC
0,MLP,1.0000,1.0000,nan
1,HGBT,1.0000,1.0000,nan


...

,Model,Accuracy,F1,AUC
0,Logistic Regression,0.7143,0.7200,0.7338
1,MLP,0.6939,0.6809,0.7342
2,HGBT,0.6429,0.6237,0.6971


--- Task: sentences_cleaned ---
Class Distribution (Speakers): {'HC': 45, 'PD': 39}
Feature Dimension: 64


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.6548,0.6420,0.6638
1,MLP,0.7500,0.7273,0.7704
2,HGBT,0.7500,0.7200,0.7909


--- Task: words_merged ---
Class Distribution (Speakers): {'HC': 45, 'PD': 41}
Feature Dimension: 54


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.7791,0.7654,0.8623
1,MLP,0.8140,0.8049,0.8650
2,HGBT,0.7093,0.6753,0.7724


## Classification with Sex as a Feature
In this section, we repeat the experiments but include the speaker's **sex** as an additional feature in the input vector.
This helps determine if combining acoustic phoneme statistics with demographic information improves the detection of Parkinson's Disease.

In [6]:
def run_experiment_with_sex(df, title):
    # Create a copy and encode sex as a numeric feature
    df_sex = df.copy()
    df_sex['sex_binary'] = df_sex['sex'].map({'M': 0, 'F': 1})
    
    X = df_sex.drop(columns=['status', 'sex'])
    y = df_sex['status'].apply(lambda x: 1 if x == 'PD' else 0)
    groups = df_sex.index
    
    counts = df_sex['status'].value_counts().to_dict()
    print(f'--- {title} (with Sex feature) ---')
    print(f'Class Distribution: {counts}')
    print(f'Feature Dimension: {X.shape[1]} (Phoneme stats + Sex)')
    
    # SAFETY CHECK: Skip if only one class is present
    if len(counts) < 2:
        print(f'Skipping {title}: Only one class present in this subset.')
        return None
    
    n_splits = max(2, min(10, min(counts.values())))
    cv = StratifiedGroupKFold(n_splits=n_splits)
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'MLP': MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500, random_state=42),
        'HGBT': HistGradientBoostingClassifier(random_state=42)
    }
    
    results = []
    for name, model in models.items():
        pipeline = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        y_true, y_pred, y_prob = [], [], []
        for train_idx, test_idx in cv.split(X, y, groups=groups):
            pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
            y_true.extend(y.iloc[test_idx])
            y_pred.extend(pipeline.predict(X.iloc[test_idx]))
            p = pipeline.predict_proba(X.iloc[test_idx])[:, 1] if hasattr(model, 'predict_proba') else pipeline.predict(X.iloc[test_idx])
            y_prob.extend(p)
        results.append({'Model': name, 'Accuracy': accuracy_score(y_true, y_pred), 'F1': f1_score(y_true, y_pred), 'AUC': roc_auc_score(y_true, y_prob)})
    
    res_df = pd.DataFrame(results)
    display(res_df.style.format(precision=4))
    print('\\n')
    return res_df

# Global Experiment with Sex
run_experiment_with_sex(feature_df, 'All Tasks Combined')

# Per-Task Experiments with Sex
for task in TASKS:
    df_task = df_all[df_all['task'] == task]
    if df_task.empty: continue
    feat_task = get_speaker_features(df_task)
    if len(feat_task) < 20: continue
    run_experiment_with_sex(feat_task, f'Task: {task}')

--- All Tasks Combined (with Sex feature) ---
Class Distribution: {'PD': 50, 'HC': 50}
Feature Dimension: 67 (Phoneme stats + Sex)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.8100,0.8081,0.8888
1,MLP,0.7900,0.7835,0.8964
2,HGBT,0.7600,0.7600,0.8344


\n
--- Task: ddk (with Sex feature) ---
Class Distribution: {'PD': 50}
Feature Dimension: 27 (Phoneme stats + Sex)
Skipping Task: ddk: Only one class present in this subset.
--- Task: monologue_split (with Sex feature) ---
Class Distribution: {'PD': 50, 'HC': 50}
Feature Dimension: 67 (Phoneme stats + Sex)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.6300,0.6022,0.6572
1,MLP,0.6100,0.5895,0.7036
2,HGBT,0.6600,0.6731,0.7012


\n
--- Task: readtext_split (with Sex feature) ---
Class Distribution: {'HC': 50, 'PD': 48}
Feature Dimension: 57 (Phoneme stats + Sex)


...

\n
--- Task: sentences_cleaned (with Sex feature) ---
Class Distribution: {'HC': 45, 'PD': 39}
Feature Dimension: 65 (Phoneme stats + Sex)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.6310,0.6265,0.6684
1,MLP,0.7500,0.7273,0.7801
2,HGBT,0.7619,0.7368,0.7932


\n
--- Task: words_merged (with Sex feature) ---
Class Distribution: {'HC': 45, 'PD': 41}
Feature Dimension: 55 (Phoneme stats + Sex)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.7674,0.7561,0.8726
1,MLP,0.8140,0.8049,0.8802
2,HGBT,0.6860,0.6494,0.7696


\n


## Per-Speaker Detailed Summary
This section provides a detailed breakdown of the classification results for each individual speaker using the **Logistic Regression** model (acoustic features only).
- **Green rows**: Correct classification.
- **Red rows**: Misclassification.

In [7]:
def display_per_speaker_summary(df, model_name='Logistic Regression'):
    X = df.drop(columns=['status', 'sex'])
    y = df['status'].apply(lambda x: 1 if x == 'PD' else 0)
    groups = df.index
    
    # Using the same model config as earlier
    model = LogisticRegression(max_iter=1000, random_state=42)
    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    
    cv = StratifiedGroupKFold(n_splits=10)
    speaker_results = []
    
    for train_idx, test_idx in cv.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        test_speakers = groups[test_idx]
        
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)
        probs = pipeline.predict_proba(X_test)
        
        for i, sid in enumerate(test_speakers):
            speaker_results.append({
                'Speaker ID': sid,
                'Actual': 'PD' if y_test.iloc[i] == 1 else 'HC',
                'Predicted': 'PD' if preds[i] == 1 else 'HC',
                'Prob HC': probs[i][0],
                'Prob PD': probs[i][1]
            })
    
    res_df = pd.DataFrame(speaker_results).set_index('Speaker ID')
    
    # Sorting for better readability
    res_df = res_df.sort_index()
    
    # Styling function
    def color_correct(row):
        color = '#d4edda' if row['Actual'] == row['Predicted'] else '#f8d7da'
        return [f'background-color: {color}'] * len(row)
    
    display(res_df.style.apply(color_correct, axis=1).format({
        'Prob HC': '{:.4f}',
        'Prob PD': '{:.4f}'
    }))
    return res_df

# Run detailed summary for the combined dataset
summary_df = display_per_speaker_summary(feature_df)

,Actual,Predicted,Prob HC,Prob PD
Speaker ID,,,,
001PD,PD,PD,0.4206,0.5794
002PD,PD,PD,0.4689,0.5311
003PD,PD,PD,0.0000,1.0000
004PD,PD,PD,0.0016,0.9984
005PD,PD,PD,0.3445,0.6555
006PD,PD,PD,0.0232,0.9768
007PD,PD,PD,0.0728,0.9272
008PD,PD,PD,0.2910,0.7090
009PD,PD,PD,0.0109,0.9891
